<a href="https://colab.research.google.com/github/ch-manasa/AI-and-ML/blob/main/Tourism_MLOps_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


## Pre-requisites

##Installing and Importing Necessary Libraries

In [1]:
!pip install -q PyGithub==2.9.1
!pip install mlflow==3.0.1 pyngrok==7.2.12 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00


In [1]:
import os
import sys
from github import Github, GithubException

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib
import mlflow

## Configuration

In [2]:
GITHUB_USERNAME   = "ch-manasa"
REPO_NAME         = "AI-and-ML"
COLAB_SECRET_NAME = "GH_TOKEN"

REPO   = f"{GITHUB_USERNAME}/{REPO_NAME}"
BRANCH = "main"

### Secrets in Colab

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
from google.colab import userdata

os.environ["GH_TOKEN"] = userdata.get(COLAB_SECRET_NAME)
print("Token loaded.")

Token loaded.


# Model Building

In [5]:
import os

BASE_PROJECT_DIR = '/content/drive/MyDrive/AI&ML/MLOPs'

project_path = os.path.join(BASE_PROJECT_DIR, "tourism_project")
model_building_path = os.path.join(project_path, "model_building")

os.makedirs(project_path, exist_ok=True)
os.makedirs(model_building_path, exist_ok=True)

## Data Registration

In [6]:
model_building_path = os.path.join(project_path, "data")
os.makedirs(model_building_path, exist_ok=True)

In [7]:
%%writefile /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/model_building/data_register.py
import os
import sys
import pandas as pd

DATA_PATH = os.path.join("tourism_project", "data", "tourism.csv")  # relative to repo root

EXPECTED_COLUMNS = [
    "CustomerID",
    "ProdTaken",
    "Age",
    "TypeofContact",
    "CityTier",
    "DurationOfPitch",
    "Occupation",
    "Gender",
    "NumberOfPersonVisiting",
    "NumberOfFollowups",
    "ProductPitched",
    "PreferredPropertyStar",
    "MaritalStatus",
    "NumberOfTrips",
    "Passport",
    "PitchSatisfactionScore",
    "OwnCar",
    "NumberOfChildrenVisiting",
    "Designation",
    "MonthlyIncome",
]


def main():
    if not os.path.exists(DATA_PATH):
        print(f"ERROR: dataset not found at {DATA_PATH}")
        sys.exit(1)

    df = pd.read_csv(DATA_PATH)

    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing_cols:
        print(f"ERROR: dataset is missing expected columns: {missing_cols}")
        sys.exit(1)

    print("Dataset Registration Summary")
    print("=" * 40)
    print(f"File path        : {DATA_PATH}")
    print(f"Rows             : {df.shape[0]}")
    print(f"Columns          : {df.shape[1]}")
    print(f"Expected columns : all {len(EXPECTED_COLUMNS)} present")
    print(f"Missing values   :\n{df.isnull().sum().sum()} total")
    print(f"Target balance   :\n{df['ProdTaken'].value_counts(normalize=True).round(3).to_dict()}")
    print("=" * 40)
    print("Dataset registered and validated successfully.")


if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/model_building/data_register.py


In [9]:
!cd "/content/drive/MyDrive/AI&ML/MLOPs" && python tourism_project/model_building/data_register.py

Dataset Registration Summary
File path        : tourism_project/data/tourism.csv
Rows             : 4128
Columns          : 21
Expected columns : all 20 present
Missing values   :
0 total
Target balance   :
{0: 0.807, 1: 0.193}
Dataset registered and validated successfully.


## Exploratory Data Analysis (EDA)

In [21]:
import pandas as pd

df_eda = pd.read_csv("/content/drive/MyDrive/AI&ML/MLOPs/tourism_project/data/tourism.csv")
print("Shape:", df_eda.shape)
df_eda.info()

Shape: (4128, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4128 entries, 0 to 4127
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                4128 non-null   int64  
 1   CustomerID                4128 non-null   int64  
 2   ProdTaken                 4128 non-null   int64  
 3   Age                       4128 non-null   float64
 4   TypeofContact             4128 non-null   object 
 5   CityTier                  4128 non-null   int64  
 6   DurationOfPitch           4128 non-null   float64
 7   Occupation                4128 non-null   object 
 8   Gender                    4128 non-null   object 
 9   NumberOfPersonVisiting    4128 non-null   int64  
 10  NumberOfFollowups         4128 non-null   float64
 11  ProductPitched            4128 non-null   object 
 12  PreferredPropertyStar     4128 non-null   float64
 13  MaritalStatus             4128 non-null   obj

In [24]:
# Missing values and duplicates
print("Missing values:\n", df_eda.isnull().sum())
print("\nDuplicate rows:", df_eda.duplicated().sum())

Missing values:
 Unnamed: 0                  0
CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

Duplicate rows: 0


In [23]:
# Target balance
df_eda["ProdTaken"].value_counts(normalize=True).round(3)

,proportion
ProdTaken,
0,0.807
1,0.193


**Observation:**
- No missing values and no duplicate rows. The target `ProdTaken` is imbalanced - roughly 81% did not buy vs 19% who did. This is why `train.py` tunes on **F1-score** rather than accuracy, and uses `scale_pos_weight` to counter the imbalance.

In [25]:
# Check categorical columns for inconsistent labels
for c in ["TypeofContact", "Occupation", "Gender", "ProductPitched", "MaritalStatus", "Designation"]:
    print(c, "->", df_eda[c].value_counts().to_dict())

TypeofContact -> {'Self Enquiry': 2918, 'Company Invited': 1210}
Occupation -> {'Salaried': 1999, 'Small Business': 1746, 'Large Business': 381, 'Free Lancer': 2}
Gender -> {'Male': 2463, 'Female': 1510, 'Fe Male': 155}
ProductPitched -> {'Basic': 1615, 'Deluxe': 1422, 'Standard': 737, 'Super Deluxe': 250, 'King': 104}
MaritalStatus -> {'Married': 1990, 'Divorced': 789, 'Unmarried': 682, 'Single': 667}
Designation -> {'Executive': 1615, 'Manager': 1422, 'Senior Manager': 737, 'AVP': 250, 'VP': 104}


**Observation - data quality issues found:**
- `Gender` has a stray category `'Fe Male'` (155 rows) that is clearly a typo for `'Female'`.
- `MaritalStatus` has both `'Unmarried'` (682 rows) and `'Single'` (667 rows), which represent the same status recorded inconsistently.

Both are corrected in `prep.py` (`Fe Male` -> `Female`, `Unmarried` -> `Single`) before the train/test split, so the model doesn't treat them as distinct categories.

In [26]:
# Purchase rate by key categorical / numeric drivers
for c in ["ProductPitched", "Occupation", "MaritalStatus", "TypeofContact", "Passport", "CityTier"]:
    print(f"\n{c} -> purchase rate:")
    print(df_eda.groupby(c)["ProdTaken"].mean().round(3).sort_values(ascending=False))


ProductPitched -> purchase rate:
ProductPitched
Basic           0.300
Standard        0.161
Deluxe          0.116
King            0.087
Super Deluxe    0.076
Name: ProdTaken, dtype: float64

Occupation -> purchase rate:
Occupation
Free Lancer       1.000
Large Business    0.291
Small Business    0.186
Salaried          0.180
Name: ProdTaken, dtype: float64

MaritalStatus -> purchase rate:
MaritalStatus
Single       0.364
Unmarried    0.243
Married      0.142
Divorced     0.133
Name: ProdTaken, dtype: float64

TypeofContact -> purchase rate:
TypeofContact
Company Invited    0.227
Self Enquiry       0.179
Name: ProdTaken, dtype: float64

Passport -> purchase rate:
Passport
1    0.358
0    0.124
Name: ProdTaken, dtype: float64

CityTier -> purchase rate:
CityTier
2    0.265
3    0.241
1    0.165
Name: ProdTaken, dtype: float64


**Observation - key drivers of purchase:**
- **Passport holders convert almost 3x more often** than non-holders (~36% vs ~12%) - passport-ready customers are more travel-inclined and receptive to a wellness package pitch.
- **Single customers convert most (~36%)**, married/divorced customers convert least (~13-14%) - likely reflects fewer scheduling/budget constraints.
- Customers pitched the **`Basic`** package convert far more often (~30%) than those pitched `Super Deluxe` (~7.6%) - counter-intuitive at first, but consistent with `Basic` being pitched to the largest, most price-sensitive segment.
- `Designation` and `ProductPitched` show near-identical purchase-rate patterns (e.g. Executive/Basic both ~30%, AVP/SuperDeluxe both ~7.6%) - these two fields are closely tied to each other (sales reps pitch package tier based on seniority), so the model has some redundant signal here rather than two fully independent features. This isn't treated as an error - the redundancy is realistic business behavior - but it's worth calling out when interpreting feature importances later.
- `TypeofContact == "Company Invited"` converts somewhat better (~23%) than `"Self Enquiry"` (~18%), suggesting proactive company outreach mildly outperforms inbound inquiries.
- `DurationOfPitch` has a max of 127 minutes against a median of 14 - a clear outlier, but plausible (a very long consultative pitch) rather than an obvious data-entry error, so it is left untreated. XGBoost's tree splits are not sensitive to this kind of scale outlier the way a linear model would be.


## Data Preparation

The script below loads the dataset from the repository data folder, removes unnecessary columns (`Unnamed: 0`, the row index artifact, and `CustomerID`, a pure identifier), fixes two inconsistent category labels found during EDA (`Fe Male` -> `Female`, `Unmarried` -> `Single`), drops exact duplicate rows, and splits the cleaned dataset into training and testing sets, saved locally as CSV files.

In [10]:
%%writefile /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/model_building/prep.py
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths are relative to the repo root - this script runs both locally (with cwd
# set to the Drive project folder) and inside GitHub Actions (with cwd = repo root
# after actions/checkout), so no absolute Drive path is used here.
project_root_path = "tourism_project"
data_path = os.path.join(project_root_path, "data")
DATA_PATH = os.path.join(data_path, "tourism.csv")
model_building_path = os.path.join(project_root_path, "model_building")

TARGET_COL = "ProdTaken"

# Columns that carry no predictive signal (pure identifiers / row index)
DROP_COLS = ["Unnamed: 0", "CustomerID"]


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop unnecessary identifier columns if present
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

    # Fix inconsistent category labels found during EDA
    if "Gender" in df.columns:
        df["Gender"] = df["Gender"].replace({"Fe Male": "Female"})
    if "MaritalStatus" in df.columns:
        df["MaritalStatus"] = df["MaritalStatus"].replace({"Unmarried": "Single"})

    # Drop exact duplicate rows, if any
    df.drop_duplicates(inplace=True)

    return df


def main():
    os.makedirs(model_building_path, exist_ok=True)

    df = pd.read_csv(DATA_PATH)
    print(f"Loaded raw dataset: {df.shape}")

    df = clean_data(df)
    print(f"After cleaning: {df.shape}")

    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]

    Xtrain, Xtest, ytrain, ytest = train_test_split(
        X, y, test_size=0.2, random_state=1, stratify=y
    )

    Xtrain.to_csv(os.path.join(model_building_path, "Xtrain.csv"), index=False)
    Xtest.to_csv(os.path.join(model_building_path, "Xtest.csv"), index=False)
    ytrain.to_csv(os.path.join(model_building_path, "ytrain.csv"), index=False)
    ytest.to_csv(os.path.join(model_building_path, "ytest.csv"), index=False)

    print(f"Xtrain: {Xtrain.shape}, Xtest: {Xtest.shape}")
    print(f"ytrain: {ytrain.shape}, ytest: {ytest.shape}")
    print("Train/test splits saved under tourism_project/model_building/")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/model_building/prep.py


In [11]:
!cd "/content/drive/MyDrive/AI&ML/MLOPs" && python tourism_project/model_building/prep.py

Loaded raw dataset: (4128, 21)
After cleaning: (4011, 19)
Xtrain: (3208, 18), Xtest: (803, 18)
ytrain: (3208,), ytest: (803,)
Train/test splits saved under tourism_project/model_building/


## Model Training and Registration with Experimentation Tracking

## Model Training and Registration with Experimentation Tracking

**Why XGBoost, and why not compare multiple models here:** The rubric allows only any single algorithm from Decision Tree, Bagging, Random Forest, AdaBoost, Gradient Boosting, or XGBoost - it does not require building and comparing several.

XGBoost is chosen because:
- The data is tabular with a mix of numeric (`Age`, `MonthlyIncome`, `DurationOfPitch`, ...) and categorical (`Occupation`, `ProductPitched`, ...) features with non-linear, interacting effects on purchase likelihood (e.g. `Passport` x `MaritalStatus` x `ProductPitched` combinations found in the EDA above) - gradient-boosted trees capture this without manual feature crosses.
- It natively supports `scale_pos_weight`, which directly addresses the ~81/19 class imbalance found in the EDA, without needing a separate resampling step (e.g. SMOTE).
- Among the rubric-allowed options, boosted trees (Gradient Boosting / XGBoost) generally outperform a single Decision Tree or basic Bagging on this kind of moderate-size, moderately imbalanced tabular problem, and XGBoost's regularization (`max_depth`, `learning_rate`) gives finer overfitting control than plain Gradient Boosting.

**Why F1-score as the tuning metric:** Accuracy would be misleading on an 81/19 imbalanced target (a model that always predicts "no purchase" would already score ~81% accuracy while being useless). F1-score balances precision and recall on the minority "will purchase" class, which is the class the marketing team actually cares about targeting.



- The script below loads the train/test splits produced by the previous job, defines an **XGBoost classifier** wrapped in a preprocessing pipeline (`StandardScaler` for numeric features, `OneHotEncoder` for categorical features), tunes it with `GridSearchCV` over `n_estimators`, `max_depth`, `learning_rate`, and `scale_pos_weight`, logs the tuned parameters and evaluation metrics to MLflow for experiment tracking, evaluates the best model on the test set, and saves it into `tourism_project/deployment/` so the workflow can commit it into the repo.

In [33]:
%%writefile /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/model_building/train.py

import os
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)
from xgboost import XGBClassifier

# Paths are relative to the repo root - this script runs both locally (with cwd
# set to the Drive project folder) and inside GitHub Actions (with cwd = repo root
# after actions/checkout), so no absolute Drive path is used here.
project_root_path = "tourism_project"
model_building_path = os.path.join(project_root_path, "model_building")
MODEL_OUT_DIR = os.path.join(project_root_path, "deployment")
MODEL_OUT_PATH = os.path.join(MODEL_OUT_DIR, "best_model_v1.joblib")


def load_splits():
    Xtrain = pd.read_csv(os.path.join(model_building_path, "Xtrain.csv"))
    Xtest = pd.read_csv(os.path.join(model_building_path, "Xtest.csv"))
    ytrain = pd.read_csv(os.path.join(model_building_path, "ytrain.csv")).squeeze("columns")
    ytest = pd.read_csv(os.path.join(model_building_path, "ytest.csv")).squeeze("columns")
    return Xtrain, Xtest, ytrain, ytest


def build_pipeline(X: pd.DataFrame) -> Pipeline:
    categorical_features = X.select_dtypes(include="object").columns.tolist()
    numerical_features = X.select_dtypes(exclude="object").columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", XGBClassifier(
                random_state=1,
                eval_metric="logloss",
                verbosity=0,
            )),
        ]
    )
    return pipeline


def main():
    os.makedirs(MODEL_OUT_DIR, exist_ok=True)

    Xtrain, Xtest, ytrain, ytest = load_splits()
    print(f"Xtrain: {Xtrain.shape}, Xtest: {Xtest.shape}")

    # Handle class imbalance (roughly 80/20 split of ProdTaken)
    neg, pos = np.bincount(ytrain)
    scale_pos_weight = neg / pos

    pipeline = build_pipeline(Xtrain)

    param_grid = {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5],
        "model__learning_rate": [0.05, 0.1],
        "model__scale_pos_weight": [1, scale_pos_weight],
    }

    mlflow.set_experiment("tourism-wellness-package-prediction")

    with mlflow.start_run():
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring="f1",
            cv=3,
            n_jobs=1,
        )
        grid.fit(Xtrain, ytrain)

        best_model = grid.best_estimator_
        print("Best params:", grid.best_params_)

        # Log all tuned parameters to MLflow
        mlflow.log_params(grid.best_params_)

        # Evaluate on the test set
        preds = best_model.predict(Xtest)
        probs = best_model.predict_proba(Xtest)[:, 1]

        metrics = {
            "accuracy": accuracy_score(ytest, preds),
            "precision": precision_score(ytest, preds),
            "recall": recall_score(ytest, preds),
            "f1": f1_score(ytest, preds),
            "roc_auc": roc_auc_score(ytest, probs),
        }
        mlflow.log_metrics(metrics)

        best_model = grid.best_estimator_

        mlflow.log_metric(
            "best_cv_f1",
            grid.best_score_
        )
        print(f"Best Cross Validation F1 Score: {grid.best_score_:.4f}")

        print("Test set performance:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        print("\nClassification report:\n", classification_report(ytest, preds))

        report = classification_report(
            ytest,
            preds,
            output_dict=True
        )


        mlflow.log_dict(
            report,
            "classification_report.json"
        )

        results = pd.DataFrame(grid.cv_results_)

        results.to_csv(
            os.path.join(
                MODEL_OUT_DIR,
                "gridsearch_results.csv"
            ),
            index=False
        )

        mlflow.log_artifact(
            os.path.join(
                MODEL_OUT_DIR,
                "gridsearch_results.csv"
            )
        )

        print("\nTop 5 GridSearch Results:")
        print(
            results[
                [
                    "mean_test_score",
                    "param_model__n_estimators",
                    "param_model__max_depth",
                    "param_model__learning_rate",
                    "param_model__scale_pos_weight",
                ]
            ]
            .sort_values(by="mean_test_score", ascending=False)
            .head()
        )

        best_params_df = pd.DataFrame(
        grid.best_params_.items(),
        columns=["Hyperparameter", "Best Value"]
        )

        print("\nBest Hyperparameters")
        print(best_params_df)

    joblib.dump(best_model, MODEL_OUT_PATH)
    print(f"Best model saved to {MODEL_OUT_PATH}")

    with mlflow.start_run(run_id=mlflow.last_active_run().info.run_id):
        mlflow.log_artifact(MODEL_OUT_PATH, artifact_path="model")


if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/model_building/train.py


In [34]:
!cd "/content/drive/MyDrive/AI&ML/MLOPs" && python tourism_project/model_building/train.py

Xtrain: (3208, 18), Xtest: (803, 18)
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__scale_pos_weight': np.float64(4.182552504038772)}
Best Cross Validation F1 Score: 0.7418
Test set performance:
  accuracy: 0.9215
  precision: 0.7738
  recall: 0.8387
  f1: 0.8050
  roc_auc: 0.9333

Classification report:
               precision    recall  f1-score   support

           0       0.96      0.94      0.95       648
           1       0.77      0.84      0.80       155

    accuracy                           0.92       803
   macro avg       0.87      0.89      0.88       803
weighted avg       0.92      0.92      0.92       803


Top 5 GridSearch Results:
    mean_test_score  ...  param_model__scale_pos_weight
15         0.741842  ...                       4.182553
13         0.703174  ...                       4.182553
7          0.689817  ...                       4.182553
14         0.681315  ...                       1.000000
5   

Observation:
- The tuned XGBoost model achieves a test F1-score of 0.805 and ROC-AUC of 0.933 on the minority (purchase) class, correctly identifying ~84% of actual buyers (recall) while keeping precision at ~77%.

- This is a strong result for the business use case - the marketing team can use the model's predictions to prioritize outreach toward customers most likely to convert, substantially narrowing the pool versus contacting everyone.

# Deployment

In [14]:
BASE_PROJECT_DIR = '/content/drive/MyDrive/AI&ML/MLOPs'
project_root_path = os.path.join(BASE_PROJECT_DIR, "tourism_project")
MODEL_OUT_DIR = os.path.join(project_root_path, "deployment")
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

## Streamlit App

The app loads the model committed to `tourism_project/deployment/` by the pipeline, collects the customer/pitch inputs into a dataframe, and displays the prediction along with a confidence score.

In [15]:
%%writefile "/content/drive/MyDrive/AI&ML/MLOPs/tourism_project/deployment/app.py"
import streamlit as st
import pandas as pd
import joblib
import os

# Load the trained model pipeline committed to the repo by the MLOps pipeline
MODEL_PATH = os.path.join(os.path.dirname(__file__), "best_model_v1.joblib")
model = joblib.load(MODEL_PATH)

st.title("Visit with Us — Wellness Tourism Package Predictor")
st.write(
    "This app predicts whether a customer is likely to purchase the new "
    "Wellness Tourism Package, based on customer profile and past sales-pitch "
    "interaction data."
)

st.header("Customer Details")
col1, col2 = st.columns(2)

with col1:
    Age = st.slider("Age", 18, 65, 35)
    TypeofContact = st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"])
    CityTier = st.selectbox("City Tier", [1, 2, 3])
    Occupation = st.selectbox("Occupation", ["Salaried", "Free Lancer", "Small Business", "Large Business"])
    Gender = st.selectbox("Gender", ["Male", "Female"])
    NumberOfPersonVisiting = st.slider("Number of Persons Visiting", 1, 5, 2)
    PreferredPropertyStar = st.selectbox("Preferred Property Star", [3.0, 4.0, 5.0])
    MaritalStatus = st.selectbox("Marital Status", ["Single", "Married", "Divorced"])

with col2:
    NumberOfTrips = st.slider("Average Number of Trips per Year", 0, 10, 3)
    Passport = st.selectbox("Holds a Passport?", ["Yes", "No"])
    OwnCar = st.selectbox("Owns a Car?", ["Yes", "No"])
    NumberOfChildrenVisiting = st.slider("Number of Children Visiting (below 5 yrs)", 0, 3, 0)
    Designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    MonthlyIncome = st.number_input("Monthly Income", min_value=1000, max_value=100000, value=22000, step=500)

st.header("Sales Pitch Interaction")
col3, col4 = st.columns(2)
with col3:
    ProductPitched = st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"])
    DurationOfPitch = st.slider("Duration of Pitch (minutes)", 5, 60, 15)
with col4:
    NumberOfFollowups = st.slider("Number of Follow-ups", 0, 10, 4)
    PitchSatisfactionScore = st.selectbox("Pitch Satisfaction Score", [1, 2, 3, 4, 5])

input_data = pd.DataFrame([{
    "Age": Age,
    "TypeofContact": TypeofContact,
    "CityTier": CityTier,
    "DurationOfPitch": DurationOfPitch,
    "Occupation": Occupation,
    "Gender": Gender,
    "NumberOfPersonVisiting": NumberOfPersonVisiting,
    "NumberOfFollowups": NumberOfFollowups,
    "ProductPitched": ProductPitched,
    "PreferredPropertyStar": PreferredPropertyStar,
    "MaritalStatus": MaritalStatus,
    "NumberOfTrips": NumberOfTrips,
    "Passport": 1 if Passport == "Yes" else 0,
    "PitchSatisfactionScore": PitchSatisfactionScore,
    "OwnCar": 1 if OwnCar == "Yes" else 0,
    "NumberOfChildrenVisiting": NumberOfChildrenVisiting,
    "Designation": Designation,
    "MonthlyIncome": MonthlyIncome,
}])

if st.button("Predict", type="primary"):
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    if prediction == 1:
        st.success(f"Likely to purchase the Wellness Package (confidence: {probability:.1%})")
    else:
        st.warning(f"Unlikely to purchase the Wellness Package (confidence: {1 - probability:.1%})")

Writing /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/deployment/app.py


## App Dependencies

In [16]:
%%writefile "/content/drive/MyDrive/AI&ML/MLOPs/tourism_project/deployment/requirements.txt"
streamlit==1.43.2
pandas==2.2.2
scikit-learn==1.6.1
xgboost==2.1.4
joblib==1.4.2

Writing /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/deployment/requirements.txt


# MLOps Pipeline with Github Actions Workflow

In [17]:
os.makedirs(".github/workflows", exist_ok=True)

In [18]:
%%writefile .github/workflows/pipeline.yml
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch
  workflow_dispatch:  # lets you click "Run workflow" in the Actions tab

permissions:
  contents: write

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Register Dataset
        run: python tourism_project/model_building/data_register.py
      - name: Upload Registered Dataset
        uses: actions/upload-artifact@v4
        with:
          name: registered-data
          path: tourism_project/data/tourism.csv

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Run Data Preparation
        run: python tourism_project/model_building/prep.py
      - name: Upload Train Test Splits
        uses: actions/upload-artifact@v4
        with:
          name: data-splits
          path: |
            tourism_project/model_building/Xtrain.csv
            tourism_project/model_building/Xtest.csv
            tourism_project/model_building/ytrain.csv
            tourism_project/model_building/ytest.csv

  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install Dependencies
        run: pip install -r tourism_project/requirements.txt
      - name: Download Train Test Splits
        uses: actions/download-artifact@v4
        with:
          name: data-splits
          path: tourism_project/model_building # Download artifacts to this path
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        run: python tourism_project/model_building/train.py
      - name: Commit Trained Model
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add tourism_project/deployment/best_model_v1.joblib
          git commit -m "Add trained model [skip ci]" || echo "No changes to commit"
          git push

Writing .github/workflows/pipeline.yml


## Requirements file for the Github Actions Workflow

In [19]:
%%writefile /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
xgboost==2.1.4
joblib==1.4.2
mlflow==3.0.1

Writing /content/drive/MyDrive/AI&ML/MLOPs/tourism_project/requirements.txt


## Github Authentication and Push Files

In [20]:
CSV_LOCAL = "/content/drive/MyDrive/AI&ML/MLOPs/tourism_project/data/tourism.csv"

assert os.path.exists(CSV_LOCAL), (
    f"{CSV_LOCAL} not found. Please upload tourism.csv into the "
    "tourism_project/data/ folder before running this cell."
)

# Connect to GitHub
gh = Github(os.environ["GH_TOKEN"])
user = gh.get_user()
REPO = f"{user.login}/{REPO_NAME}"

try:
    repo = gh.get_repo(REPO)
    print("Repository already exists:", repo.full_name)
except GithubException:
    repo = user.create_repo(
        REPO_NAME,
        private=False,
        auto_init=True
    )
    print("Repository created:", repo.full_name)


def push_folder(local_dir, repo_prefix):
    for root, _, files in os.walk(local_dir):
        for fname in files:
            local_path = os.path.join(root, fname)
            rel_path = os.path.relpath(local_path, local_dir)
            repo_path = os.path.join(repo_prefix, rel_path).replace(os.sep, "/")
            content = open(local_path, "rb").read()

            try:
                sha = repo.get_contents(repo_path, ref=BRANCH).sha
                repo.update_file(
                    repo_path,
                    f"update {repo_path}",
                    content,
                    sha,
                    branch=BRANCH
                )
                print("updated", repo_path)

            except GithubException:
                repo.create_file(
                    repo_path,
                    f"add {repo_path}",
                    content,
                    branch=BRANCH
                )
                print("added  ", repo_path)

push_folder(os.path.join(BASE_PROJECT_DIR, "tourism_project"), "tourism_project")

push_folder(".github", ".github")

# --- Verify that all required files exist in the repository ---
print("\nVerifying repository structure...")

required = [
    "tourism_project/requirements.txt",
    "tourism_project/model_building/data_register.py",
    "tourism_project/model_building/prep.py",
    "tourism_project/model_building/train.py",
    "tourism_project/data/tourism.csv",
    "tourism_project/deployment/app.py",
    "tourism_project/deployment/requirements.txt",
    ".github/workflows/pipeline.yml",
]

all_ok = True

for file in required:
    try:
        repo.get_contents(file, ref=BRANCH)
        print("\u2713", file)
    except GithubException:
        print("\u2717 Missing:", file)
        all_ok = False

if all_ok:
    print("\nAll project files have been successfully pushed to GitHub.")
else:
    print("\nSome files are missing. Please review the messages above.")

/tmp/ipykernel_2951/1877163260.py:9: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  gh = Github(os.environ["GH_TOKEN"])


Repository already exists: ch-manasa/AI-and-ML
added   tourism_project/requirements.txt
added   tourism_project/model_building/data_register.py
added   tourism_project/model_building/prep.py
added   tourism_project/model_building/Xtrain.csv
added   tourism_project/model_building/Xtest.csv
added   tourism_project/model_building/ytrain.csv
added   tourism_project/model_building/ytest.csv
added   tourism_project/model_building/train.py
added   tourism_project/data/tourism.csv
added   tourism_project/deployment/best_model_v1.joblib
added   tourism_project/deployment/app.py
added   tourism_project/deployment/requirements.txt
updated .github/workflows/pipeline.yml

Verifying repository structure...
✓ tourism_project/requirements.txt
✓ tourism_project/model_building/data_register.py
✓ tourism_project/model_building/prep.py
✓ tourism_project/model_building/train.py
✓ tourism_project/data/tourism.csv
✓ tourism_project/deployment/app.py
✓ tourism_project/deployment/requirements.txt
✓ .github/wor

# Deploy the App on Streamlit Community Cloud


# Output Evaluation

- **GitHub** (link to repository, screenshot of folder structure and executed workflow)
- Note: Addding the screenshots link which are uploaded to github as their metadata is too large which increases the notebook size.

Github Link: https://github.com/ch-manasa/AI-and-ML/tree/main/tourism_project

workflow pipeline: https://github.com/ch-manasa/AI-and-ML/blob/main/.github/workflows/pipeline.yml

Folder structure screenshot:
https://github.com/ch-manasa/AI-and-ML/blob/main/tourism_project/images/Github%20File%20structure.png

Executed workflow screenshots:
https://github.com/ch-manasa/AI-and-ML/blob/main/tourism_project/images/Github%20workflow.png



- **Streamlit Community Cloud** (link to the Streamlit app, screenshot of Streamlit app)

Streamlit link: https://mlopstourismproject.streamlit.app/

Streamlit Scenario where chances of purchasing the program is > 80%:
https://github.com/ch-manasa/AI-and-ML/blob/main/tourism_project/images/UI%201.png

https://github.com/ch-manasa/AI-and-ML/blob/main/tourism_project/images/UI%202.png

Streamlit Scenario where chances of purchasing the program is < 70%:
https://github.com/ch-manasa/AI-and-ML/blob/main/tourism_project/images/UI%203.png

https://github.com/ch-manasa/AI-and-ML/blob/main/tourism_project/images/UI%204.png

# Observations and Business Recommendations

**Data & modeling summary:**
- The raw dataset (4,128 rows) had no missing values but two inconsistent category labels (`Gender`'s `'Fe Male'`, `MaritalStatus`'s `'Unmarried'`), which were corrected in `prep.py`. `Unnamed: 0` (row index artifact) and `CustomerID` (pure identifier) were dropped as they carry no generalizable signal.
- The target (`ProdTaken`) is imbalanced (~81% no-purchase / ~19% purchase), so the model was tuned on F1-score with `scale_pos_weight` rather than accuracy.
- The tuned XGBoost model achieves **F1 = 0.805** and **ROC-AUC = 0.933** on the held-out test set, correctly identifying ~84% of actual buyers (recall) while keeping precision at ~77% - a strong basis for prioritizing sales outreach.

**Actionable business recommendations:**
1. **Prioritize passport holders and single customers** in outbound campaigns for the Wellness Package - both segments show 2-3x higher historical conversion than the base rate, and the model's predicted-probability score can be used to rank leads within these segments too.
2. **Favor company-invited outreach over waiting on self-enquiry** where budget allows - company-invited contacts converted at ~23% vs ~18% for self-enquiry in the historical data.
3. **Re-examine the `Super Deluxe`/`King` pitch strategy** - these premium tiers convert far less often (7-9%) than `Basic`/`Standard` (16-30%); either the pricing/positioning needs adjustment for this tier, or reps should qualify customers more carefully before pitching it.
4. **Operationalize the model as a pre-call scoring tool**: before a sales rep contacts a customer, run their profile through the deployed Streamlit app (or a batch scoring job) to get a purchase-likelihood score, and focus live follow-up effort on the highest-scoring leads rather than contacting the full list uniformly.
5. **Retrain periodically** as new pitch outcomes accumulate (the GitHub Actions pipeline already automates this on every push to `main`), so the model keeps adapting to evolving customer behavior rather than going stale.